# OCR Engine Comparison on Mortgage Documents

## Objective
Compares Tesseract, EasyOCR, and PaddleOCR on mortgage/loan worksheet PDFs to evaluate extraction quality for downstream document AI tasks.

## Approach
- Install and import OCR dependencies
- Convert the PDF into page images
- Run each OCR engine on the same pages
- Save outputs and comparison notes for review

## Expected Result
This notebook supports the extraction stage of the externship pipeline by identifying which OCR engine is most useful for mortgage-style documents.

## Running in Google Colab
These notebooks were developed in Google Colab. For reproducibility, place required PDFs in a `data/` folder when running locally, or upload them to the Colab working directory. The helper functions below try common Colab and GitHub-style paths.

## Security Note
API keys are not stored in the notebook. Use Colab Secrets with the name `GOOGLE_API_KEY` or set the environment variable manually.

## Project Context
This notebook is part of a curated document intelligence externship portfolio project completed through Outamation. The work focuses on OCR, document parsing, retrieval, LLM-based question answering, and prototype application development for mortgage-style document analysis.

## Data Note
The notebooks were originally developed in Google Colab. Any document files used for testing should be placed in the `data/` folder or uploaded directly into the Colab runtime. The sample documents used for this educational project do not contain sensitive personal information.


In [ ]:
# -------------------------
# PORTABLE COLAB/GITHUB HELPERS
# -------------------------
from pathlib import Path
import os

def resolve_path(filename_or_path):
    """Find a file in common Colab and GitHub project locations."""
    candidates = [
        Path(filename_or_path),
        Path("/content") / filename_or_path,
        Path("data") / Path(filename_or_path).name,
        Path("/content/data") / Path(filename_or_path).name,
    ]
    for path in candidates:
        if path.exists():
            return str(path)
    # Return GitHub-style path as the default so users know where to place data.
    return str(Path("data") / Path(filename_or_path).name)

def get_google_api_key():
    """Load GOOGLE_API_KEY from Colab Secrets or environment variables."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return os.getenv("GOOGLE_API_KEY")


# =========================================================
# OCR Comparison on Mortgage PDF
# Tesseract vs EasyOCR vs PaddleOCR
# File: data/LenderFeesWorksheetNew (2).pdf
# =========================================================

# ------------------------------
# 1) Install dependencies
# ------------------------------
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr
!pip install -q pdf2image pytesseract easyocr paddleocr paddlepaddle opencv-python pillow matplotlib

# ------------------------------
# 2) Imports
# ------------------------------
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pdf2image import convert_from_path
import pytesseract
import easyocr
from paddleocr import PaddleOCR

# ------------------------------
# 3) File paths and folders
# ------------------------------
pdf_path = resolve_path("LenderFeesWorksheetNew (2).pdf")
output_dir = "outputs/ocr_outputs"
image_dir = os.path.join(output_dir, "/content/sample_data")
annotated_dir = os.path.join(output_dir, "/content/sample_data")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(image_dir, exist_ok=True)
os.makedirs(annotated_dir, exist_ok=True)

# ------------------------------
# 4) Convert PDF to image(s)
# ------------------------------
pages = convert_from_path(pdf_path, dpi=300)

image_paths = []
for i, page in enumerate(pages):
    img_path = os.path.join(image_dir, f"page_{i+1}.png")
    page.save(img_path, "PNG")
    image_paths.append(img_path)

print(f"Converted {len(image_paths)} page(s) from PDF.")

# ------------------------------
# 5) Utility functions
# ------------------------------
def show_image(title, img_bgr, figsize=(14, 18)):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis("off")
    plt.show()

def preprocess_for_ocr(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    denoised = cv2.medianBlur(gray, 3)
    thresh = cv2.threshold(
        denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]
    return thresh

def save_text(text, filename):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)

def save_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def preview_text_block(title, text, limit=2000):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    print(text[:limit] if text else "[No text extracted]")
    print()

# ------------------------------
# 6) Initialize OCR engines
# ------------------------------
print("Loading EasyOCR...")
easy_reader = easyocr.Reader(['en'], gpu=False)

print("Loading PaddleOCR...")
# Keep use_angle_cls in constructor, but do NOT pass cls=True to .ocr()
paddle_reader = PaddleOCR(use_angle_cls=True, lang='en')

print("All OCR engines loaded.")

# ------------------------------
# 7) Tesseract OCR
# ------------------------------
def run_tesseract(image_path, page_num):
    image_bgr = cv2.imread(image_path)
    processed = preprocess_for_ocr(image_bgr)

    text = pytesseract.image_to_string(processed, config="--oem 3 --psm 6")

    data = pytesseract.image_to_data(
        processed,
        output_type=pytesseract.Output.DICT,
        config="--oem 3 --psm 6"
    )

    annotated = image_bgr.copy()
    results = []

    n = len(data["text"])
    for i in range(n):
        txt = str(data["text"][i]).strip()
        conf = data["conf"][i]

        try:
            conf_val = float(conf)
        except:
            conf_val = -1

        if txt and conf_val > 0:
            x = int(data["left"][i])
            y = int(data["top"][i])
            w = int(data["width"][i])
            h = int(data["height"][i])

            results.append({
                "text": txt,
                "confidence": conf_val,
                "bbox": [x, y, x + w, y + h]
            })

            cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)

    save_text(text, os.path.join(output_dir, f"tesseract_page_{page_num}.txt"))
    save_json(results, os.path.join(output_dir, f"tesseract_page_{page_num}.json"))
    cv2.imwrite(os.path.join(annotated_dir, f"tesseract_page_{page_num}.png"), annotated)

    return text, results, annotated

# ------------------------------
# 8) EasyOCR
# ------------------------------
def run_easyocr(image_path, page_num):
    image_bgr = cv2.imread(image_path)
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    raw_results = easy_reader.readtext(rgb, detail=1)
    annotated = image_bgr.copy()

    extracted_lines = []
    structured = []

    for item in raw_results:
        box, text, conf = item
        extracted_lines.append(text)

        pts = np.array(box, dtype=np.int32)
        cv2.polylines(annotated, [pts], isClosed=True, color=(255, 0, 0), thickness=2)

        x_coords = [p[0] for p in box]
        y_coords = [p[1] for p in box]

        structured.append({
            "text": text,
            "confidence": float(conf),
            "bbox": [
                int(min(x_coords)),
                int(min(y_coords)),
                int(max(x_coords)),
                int(max(y_coords))
            ]
        })

    full_text = "\n".join(extracted_lines)

    save_text(full_text, os.path.join(output_dir, f"easyocr_page_{page_num}.txt"))
    save_json(structured, os.path.join(output_dir, f"easyocr_page_{page_num}.json"))
    cv2.imwrite(os.path.join(annotated_dir, f"easyocr_page_{page_num}.png"), annotated)

    return full_text, structured, annotated

# ------------------------------
# 9) PaddleOCR
# ------------------------------
def parse_paddle_results(raw_results):
    """
    Handles common PaddleOCR result formats across versions.
    Returns list of {text, confidence, bbox}.
    """
    parsed = []

    if raw_results is None:
        return parsed

    # Older format:
    # raw_results = [ [ [box], (text, conf) ], ... ]
    if isinstance(raw_results, list) and len(raw_results) > 0:
        first = raw_results[0]

        if isinstance(first, list) and len(first) > 0:
            maybe_line = first[0]

            # old line format
            if isinstance(maybe_line, list) and len(maybe_line) >= 2:
                for line in first:
                    try:
                        box = line[0]
                        text = line[1][0]
                        conf = line[1][1]
                        parsed.append({
                            "text": text,
                            "confidence": float(conf),
                            "bbox_quad": box
                        })
                    except:
                        pass
                return parsed

    # Newer formats: try deeper traversal
    def walk(obj):
        if isinstance(obj, dict):
            # Look for possible text/score/box keys
            text_keys = ["text", "rec_text", "transcription"]
            score_keys = ["score", "confidence", "rec_score"]
            box_keys = ["box", "bbox", "points", "poly"]

            found_text = None
            found_score = None
            found_box = None

            for k in text_keys:
                if k in obj:
                    found_text = obj[k]
                    break
            for k in score_keys:
                if k in obj:
                    found_score = obj[k]
                    break
            for k in box_keys:
                if k in obj:
                    found_box = obj[k]
                    break

            if found_text is not None and found_box is not None:
                parsed.append({
                    "text": str(found_text),
                    "confidence": float(found_score) if found_score is not None else -1.0,
                    "bbox_quad": found_box
                })

            for v in obj.values():
                walk(v)

        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    walk(raw_results)
    return parsed

def run_paddleocr(image_path, page_num):
    image_bgr = cv2.imread(image_path)
    annotated = image_bgr.copy()

    # Try version-safe calls
    raw_results = None

    try:
        raw_results = paddle_reader.ocr(image_path)
    except Exception as e1:
        print(f"Paddle .ocr() failed on page {page_num}: {e1}")
        try:
            raw_results = paddle_reader.predict(image_path)
        except Exception as e2:
            print(f"Paddle .predict() also failed on page {page_num}: {e2}")
            raw_results = None

    parsed_results = parse_paddle_results(raw_results)

    extracted_lines = []
    structured = []

    for item in parsed_results:
        text = item["text"]
        conf = item["confidence"]
        box = item["bbox_quad"]

        try:
            pts = np.array(box, dtype=np.int32)
            if len(pts.shape) == 2 and pts.shape[1] == 2:
                cv2.polylines(annotated, [pts], isClosed=True, color=(0, 0, 255), thickness=2)

                x_coords = pts[:, 0]
                y_coords = pts[:, 1]

                structured.append({
                    "text": text,
                    "confidence": float(conf),
                    "bbox": [
                        int(np.min(x_coords)),
                        int(np.min(y_coords)),
                        int(np.max(x_coords)),
                        int(np.max(y_coords))
                    ]
                })
                extracted_lines.append(text)
        except:
            pass

    full_text = "\n".join(extracted_lines)

    save_text(full_text, os.path.join(output_dir, f"paddleocr_page_{page_num}.txt"))
    save_json(structured, os.path.join(output_dir, f"paddleocr_page_{page_num}.json"))
    cv2.imwrite(os.path.join(annotated_dir, f"paddleocr_page_{page_num}.png"), annotated)

    return full_text, structured, annotated

# ------------------------------
# 10) Run all OCR engines
# ------------------------------
all_results = {
    "tesseract": {},
    "easyocr": {},
    "paddleocr": {}
}

for idx, image_path in enumerate(image_paths, start=1):
    print(f"\nProcessing page {idx}...\n")

    # Tesseract
    t_text, t_boxes, t_img = run_tesseract(image_path, idx)
    all_results["tesseract"][f"page_{idx}"] = {
        "text": t_text,
        "boxes": t_boxes
    }

    # EasyOCR
    e_text, e_boxes, e_img = run_easyocr(image_path, idx)
    all_results["easyocr"][f"page_{idx}"] = {
        "text": e_text,
        "boxes": e_boxes
    }

    # PaddleOCR
    p_text, p_boxes, p_img = run_paddleocr(image_path, idx)
    all_results["paddleocr"][f"page_{idx}"] = {
        "text": p_text,
        "boxes": p_boxes
    }

    # Show annotated images
    show_image(f"Tesseract - Page {idx}", t_img)
    show_image(f"EasyOCR - Page {idx}", e_img)
    show_image(f"PaddleOCR - Page {idx}", p_img)

# Save combined JSON
save_json(all_results, os.path.join(output_dir, "all_ocr_results.json"))
print("Saved combined OCR results.")

# ------------------------------
# 11) Preview raw text
# ------------------------------
for engine in ["tesseract", "easyocr", "paddleocr"]:
    for page_key, page_data in all_results[engine].items():
        preview_text_block(f"{engine.upper()} | {page_key}", page_data["text"], limit=2500)

# ------------------------------
# 12) Keyword checks for important mortgage fields
# ------------------------------
keywords = [
    "XYZ Lender",
    "380,000",
    "4.250",
    "1,869.37",
    "95,641.53",
    "John Q. Smith",
    "Mary A. Smith",
    "Hazard Insurance Premium",
    "Daily Interest Charges",
    "FEES WORKSHEET"
]

print("\n" + "=" * 90)
print("KEYWORD CHECK")
print("=" * 90)

keyword_summary = {}

for engine in ["tesseract", "easyocr", "paddleocr"]:
    combined_text = "\n".join(page_data["text"] for page_data in all_results[engine].values())
    keyword_summary[engine] = {}

    print(f"\n{engine.upper()}:")
    for kw in keywords:
        found = kw.lower() in combined_text.lower()
        keyword_summary[engine][kw] = found
        print(f"  {kw}: {'FOUND' if found else 'NOT FOUND'}")

save_json(keyword_summary, os.path.join(output_dir, "keyword_check.json"))

# ------------------------------
# 13) Optional comparison notes template
# ------------------------------
summary_template = """
OCR Comparison Notes

Document used:
- data/LenderFeesWorksheetNew (2).pdf

1. Which tool captured the loan amount, lender name, or monthly interest rate most reliably?
- Tesseract:
- EasyOCR:
- PaddleOCR:

2. Did any engine group related content more intuitively?
- Tesseract:
- EasyOCR:
- PaddleOCR:

3. Were any results surprisingly bad (skipped sections, gibberish text, broken lines)?
- Tesseract:
- EasyOCR:
- PaddleOCR:

4. How usable is the output for downstream tasks like AI model training?
- Tesseract:
- EasyOCR:
- PaddleOCR:

5. Final conclusion:
- Best overall:
- Biggest surprise:
- Best for mortgage workflows:
- Setup frustrations:
- Would I combine tools or choose one:

6. Notes from this worksheet:
- Lender name:
- Loan amount:
- Interest rate:
- Monthly payment:
- Any OCR mistakes noticed:
"""

save_text(summary_template, os.path.join(output_dir, "comparison_notes_template.txt"))
print("\nSaved comparison template.")

# ------------------------------
# 14) Quick notebook guidance
# ------------------------------
print("""
What to look for in the outputs:
- Did the engine correctly capture the lender name?
- Did it correctly extract the loan amount and interest rate?
- Did it preserve the fee table structure?
- Did it break lines unnaturally or merge unrelated fields?
- Which output would be easiest to convert into JSON for downstream AI use?

Output folder:
- outputs/ocr_outputs
""")
